# L00 · The LLM RL Map in 15 Minutes

## Goal

- distinguish agent, environment, and reward
- locate PPO, DPO, GRPO, and Agentic RL
- read a change in a useful action probability

## Setup

This cell fixes CPU, seed, offline status, and the split hash first. Toy code uses deterministic CPU operations; package trainers retain their strict global default.

In [1]:
import hashlib, json, os, platform, random, sys
from pathlib import Path
os.environ.setdefault("TORCH_DEVICE_BACKEND_AUTOLOAD", "0")
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file()), None)
if ROOT is None:
    raise RuntimeError("Run this notebook inside the RL-study repository")
sys.path.insert(0, str(ROOT / "src"))
import torch
from rl_study import __version__
from rl_study.data import build_tiny_reasoning
from rl_study.runtime import resolve_device, seed_everything
# These notebooks use only deterministic CPU toy kernels.  PyTorch 2.13's global
# guard imports the full Inductor stack, so keep the package's strict default for
# trainers while avoiding that unrelated startup cost in fresh teaching kernels.
seed_everything(42, deterministic=False)
random.seed(42)
language = os.environ.get("RL_STUDY_NOTEBOOK_LANGUAGE", "ko")
resolution = resolve_device("cpu")
dataset = build_tiny_reasoning(seed=42)
config_hash = "sha256:" + hashlib.sha256(b"L00:toy:42").hexdigest()
print(f"lesson=L00 language={language} profile=toy")
print("seed=42 network_required=False deterministic_scope=seeded_cpu_toy")
print(f"python={platform.python_version()} rl_study={__version__} torch={torch.__version__}")
print(f"requested_device=cpu resolved_device={resolution.resolved} fallback_used={resolution.fallback_used}")
print(f"config_hash={config_hash} data_split_hash={dataset.split_hash}")

lesson=L00 language=en profile=toy
seed=42 network_required=False deterministic_scope=seeded_cpu_toy
python=3.12.13 rl_study=0.1.0 torch=2.13.0
requested_device=cpu resolved_device=cpu fallback_used=False
config_hash=sha256:3bd2b5bff2836ea5d5c5b1cb21352bdd5850e9e5a4c9d2fe6c83b08e2e7cebb4 data_split_hash=sha256:f238657bbf6c0a112debf7ef3ffafb452c14308dfb5ce57d9abe4f77ac1deedd


## Steps

### 1. Position and core equation

⏱ 5 min · 1/3 section · [CORE]

Position: **whole map** → probability/optimization → bandit/MDP → policy gradient/PPO → LLM alignment → Agentic RL → evaluation

$$J(\theta)=\mathbb{E}_{a\sim\pi_\theta}[r(a)]$$

RL changes an agent's future action distribution using feedback after it acts on an observation. For an LLM, an action is a token or tool call. DPO optimizes the effect from stored preference pairs, while PPO and GRPO roll out new responses and receive rewards.

```mermaid
flowchart LR
  P[Probability·optimization] --> B[Bandit·MDP]
  B --> PG[Policy gradient·PPO]
  PG --> L[LLM policy·reward]
  L --> R[RLHF·DPO·GRPO·DAPO]
  L --> A[Agentic RL]
  R --> E[Evaluation·reproducibility]
  A --> E
```

Equivalent fallback when Mermaid is unavailable:

```text
probability·optimization → bandit → MDP/Bellman → MC/TD/Q-learning → DQN
                         └→ policy gradient → actor-critic/GAE → PPO
                                                  └→ LLM policy + preference/reward
                                                       ├→ RLHF-PPO
                                                       ├→ DPO
                                                       ├→ GRPO/RLVR → DAPO
                                                       └→ Agentic RL
all paths → evaluation·reward-hacking diagnosis·reproducibility
```

### 2. Run with small numbers

⏱ 6 min · 2/3 section · [CORE]

**Predict first:** Which way will the probability of the reward-1 action move after 20 updates? Write an answer for 20 seconds, then run the cell.

<details><summary>Show answer</summary>It increases when the objective sign is correct. This is the smallest unit checked throughout the course.</details>

In [2]:
logits = torch.zeros(2, requires_grad=True)
optimizer = torch.optim.SGD([logits], lr=0.4)
reward_by_action = torch.tensor([0.0, 1.0])
probability_history = []
for _ in range(20):
    probabilities = torch.softmax(logits, dim=-1)
    loss = -(probabilities * reward_by_action).sum()
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    probability_history.append(float(probabilities[1].detach()))
print({"p_good_start": round(probability_history[0], 3),
       "p_good_end": round(probability_history[-1], 3)})

{'p_good_start': 0.5, 'p_good_end': 0.916}


### 3. Implementation anatomy

⏱ 6 min · 3/3 section · [DEEP DIVE]

**Why this implementation:** Two actions isolate the `probability → reward → gradient` loop before state and credit assignment enter. A full trainer is more realistic but makes the first failure harder to isolate.

**Common trap:** Dropping the minus sign makes the optimizer suppress the useful action. The `p_good_end > p_good_start` regression check catches it. Regression tests: `test_reinforce_sign`.

**Checkpoint:** Continue when you can explain just one printed value.

## Checks

In [3]:
assert probability_history[-1] > probability_history[0] > 0.0
print("checks=passed")

checks=passed


**Recall:** Which of PPO, DPO, and GRPO does not require new response rollouts, and why? Answer in one or two sentences.

## Mistakes I Revisit

- Assuming a finite loss proves the implementation is correct.
- Merging `terminated` with `truncated`, or prompt with action.
- Turning one tiny seed into an algorithm ranking.

## 60-Second Recap

- **Run conclusion:** The useful-action probability rose from 0.5 to 0.916. This validates only the toy objective's direction, not an algorithm ranking.
- Executable checks: `test_reinforce_sign`.
- The output is a fixed-seed toy run, not a paper-scale result.

## Next Steps

1. L01 computes the log-probability, entropy, KL, and gradients behind this probability change.
2. Break one `[CORE]` assertion and read the failure.
3. Open the package test and connect the notebook equation to its production guard.

[Implementation note](../../docs/algorithms/cards.md) · [Course map](../../docs/course-map.en.md)

## Sources

- `sutton-barto-rl2` — `docs/sources.yml`
- `ppo-2017` — `docs/sources.yml`
- `dpo-2023` — `docs/sources.yml`
- `deepseekmath-grpo-2024` — `docs/sources.yml`
- `agent-lightning-2025` — `docs/sources.yml`